# Étape 1: Modèle Skip-gram avec PyTorch

**Objectif:** Adapter le code CBOW vers un modèle Skip-gram

**Différence clé:**
- **CBOW:** Contexte → Mot cible
- **Skip-gram:** Mot cible → Contexte

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

## 1. Corpus de données

In [ ]:
# Corpus simple - vous pouvez utiliser un corpus plus large
corpus = [
    "Le Maroc est un pays situé en Afrique du Nord.",
    "Rabat est la capitale du Maroc.",
    "Le Maroc est connu pour sa culture riche.",
    "L'intelligence artificielle transforme le monde.",
    "Le machine learning est une branche de l'intelligence artificielle.",
    "Python est un langage de programmation populaire.",
    "Les réseaux de neurones sont utilisés en deep learning."
]

## 2. Prétraitement

In [ ]:
# Tokenisation et création du vocabulaire
def tokenize(corpus):
    tokens = [sentence.lower().split() for sentence in corpus]
    vocab = set([word for sentence in tokens for word in sentence])
    word2idx = {word: idx for idx, word in enumerate(vocab)}
    idx2word = {idx: word for word, idx in word2idx.items()}
    return tokens, word2idx, idx2word

tokens, word2idx, idx2word = tokenize(corpus)
vocab_size = len(word2idx)

print(f"Taille du vocabulaire: {vocab_size}")
print(f"\nPremiers mots du vocabulaire: {list(word2idx.keys())[:10]}")

## 3. Création des paires (Target → Context) pour Skip-gram

**IMPORTANT:** Contrairement à CBOW, ici on prédit le contexte à partir du mot central!

In [ ]:
def create_skipgram_pairs(tokens, window_size=2):
    """
    Créer des paires (target, context_word) pour Skip-gram
    Pour chaque mot cible, on génère une paire avec chaque mot du contexte
    """
    skipgram_pairs = []
    
    for sentence in tokens:
        for i, target in enumerate(sentence):
            # Obtenir le contexte (fenêtre autour du mot cible)
            context_start = max(0, i - window_size)
            context_end = min(len(sentence), i + window_size + 1)
            
            # Pour chaque mot dans le contexte (sauf le mot cible lui-même)
            for j in range(context_start, context_end):
                if j != i:  # Ne pas inclure le mot cible dans son propre contexte
                    context_word = sentence[j]
                    skipgram_pairs.append((target, context_word))
    
    return skipgram_pairs

def encode_skipgram_pairs(skipgram_pairs, word2idx):
    """Encoder les paires en indices"""
    encoded_pairs = []
    for target, context in skipgram_pairs:
        target_idx = word2idx[target]
        context_idx = word2idx[context]
        encoded_pairs.append((target_idx, context_idx))
    return encoded_pairs

In [ ]:
# Créer les paires Skip-gram
skipgram_pairs = create_skipgram_pairs(tokens, window_size=2)
encoded_pairs = encode_skipgram_pairs(skipgram_pairs, word2idx)

print(f"Nombre total de paires Skip-gram: {len(skipgram_pairs)}\n")
print("Exemples de paires (Target → Context):")
for i in range(min(10, len(skipgram_pairs))):
    print(f"Target: '{skipgram_pairs[i][0]}' → Context: '{skipgram_pairs[i][1]}'")

## 4. Modèle Skip-gram

In [ ]:
class SkipGramModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(SkipGramModel, self).__init__()
        # Embedding layer pour les mots
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        # Couche linéaire pour prédire le contexte
        self.linear = nn.Linear(embedding_dim, vocab_size)
    
    def forward(self, target):
        # target: [batch_size]
        embeds = self.embeddings(target)  # [batch_size, embedding_dim]
        out = self.linear(embeds)  # [batch_size, vocab_size]
        log_probs = torch.log_softmax(out, dim=1)
        return log_probs

## 5. Entraînement du modèle

In [ ]:
def train_skipgram(model, encoded_pairs, epochs=100, learning_rate=0.01):
    loss_function = nn.NLLLoss()
    optimizer = optim.SGD(model.parameters(), lr=learning_rate)
    
    losses = []
    
    for epoch in range(epochs):
        total_loss = 0
        
        for target_idx, context_idx in encoded_pairs:
            # Convertir en tenseurs
            target_var = torch.tensor([target_idx], dtype=torch.long)
            context_var = torch.tensor([context_idx], dtype=torch.long)
            
            # Forward pass
            model.zero_grad()
            log_probs = model(target_var)
            
            # Calculer la loss
            loss = loss_function(log_probs, context_var)
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        losses.append(total_loss)
        
        if epoch % 10 == 0:
            print(f'Epoch {epoch}, Loss: {total_loss:.4f}')
    
    return losses

In [ ]:
# Initialiser et entraîner le modèle
embedding_dim = 50  # Dimension des embeddings
model = SkipGramModel(vocab_size, embedding_dim)

print("Début de l'entraînement...\n")
losses = train_skipgram(model, encoded_pairs, epochs=100, learning_rate=0.01)
print("\nEntraînement terminé!")

## 6. Visualisation de la Loss

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(losses)
plt.title('Évolution de la Loss pendant l\'entraînement')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

## 7. Extraction de la matrice d'embeddings

In [ ]:
# Extraire la matrice d'embeddings
embedding_matrix = model.embeddings.weight.detach().numpy()

print(f"Taille de la matrice d'embeddings: {embedding_matrix.shape}")
print(f"Vocabulaire: {vocab_size} mots")
print(f"Dimension: {embedding_dim}")
print(f"\nPremières lignes de la matrice:\n{embedding_matrix[:5]}")

## 8. Fonctions utilitaires

In [ ]:
def get_word_vector(model, word, word2idx):
    """Obtenir le vecteur d'un mot"""
    idx = word2idx[word]
    return model.embeddings.weight[idx].detach().numpy()

def cosine_similarity(vec1, vec2):
    """Calculer la similarité cosinus entre deux vecteurs"""
    num = np.dot(vec1, vec2)
    den = np.linalg.norm(vec1) * np.linalg.norm(vec2)
    return num / den if den != 0 else 0

def find_similar_words(model, word, word2idx, idx2word, top_k=5):
    """Trouver les mots les plus similaires"""
    target_vec = get_word_vector(model, word, word2idx)
    similarities = {}
    
    for other_word in word2idx.keys():
        if other_word == word:
            continue
        vec = get_word_vector(model, other_word, word2idx)
        sim = cosine_similarity(target_vec, vec)
        similarities[other_word] = sim
    
    sorted_words = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
    return sorted_words[:top_k]

In [ ]:
# Tester la similarité
test_word = "maroc"
if test_word in word2idx:
    print(f"\nMots les plus similaires à '{test_word}':")
    similar = find_similar_words(model, test_word, word2idx, idx2word, top_k=5)
    for word, sim in similar:
        print(f"  {word}: {sim:.4f}")

## 9. Sauvegarder le modèle et les données

In [ ]:
# Sauvegarder pour utilisation dans les prochains notebooks
import pickle

# Sauvegarder le modèle
torch.save(model.state_dict(), 'skipgram_model.pth')

# Sauvegarder les dictionnaires
with open('word2idx.pkl', 'wb') as f:
    pickle.dump(word2idx, f)

with open('idx2word.pkl', 'wb') as f:
    pickle.dump(idx2word, f)

# Sauvegarder la matrice d'embeddings
np.save('embedding_matrix.npy', embedding_matrix)

print("✅ Modèle et données sauvegardés!")
print("\nFichiers créés:")
print("  - skipgram_model.pth")
print("  - word2idx.pkl")
print("  - idx2word.pkl")
print("  - embedding_matrix.npy")

## ✅ Étape 1 Complète!

**Prochaine étape:** Notebook 2 - Visualisation avec ACP